In [0]:
import sys, logging

MODULE_BASE = "/Workspace/Users/jatin.jangid.104@gmail.com/adtech/generator_src"
CONFIG_PATH = "/Workspace/Users/jatin.jangid.104@gmail.com/adtech/generator_src/config/generator_config.yaml"

if MODULE_BASE not in sys.path:
    sys.path.insert(0, MODULE_BASE)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)

print(f"MODULE_BASE : {MODULE_BASE}")
print(f"CONFIG_PATH : {CONFIG_PATH}")
print("Environment ready")

In [0]:
from core.config_loader import ConfigLoader
from validation.schema_validator import SchemaValidator

# Override output path to point at Unity Catalog Volume
OVERRIDES = {
    "platform": {
        "output_base_path": "/Volumes/main/adtech_raw/adtech-raw"
    }
}

config = ConfigLoader(CONFIG_PATH, overrides=OVERRIDES).load().config
validator = SchemaValidator(config, max_record_age_hours=72)

print(f"seed           : {config.platform.seed}")
print(f"user pool      : {config.users.total_pool_size}")
print(f"campaigns      : {config.campaigns.count}")
print(f"output path    : {config.platform.output_base_path}")
print(f"batch interval : {config.platform.batch_interval_seconds}s")
print("Config loaded")

In [0]:
import os

VOLUME_PATH = "/Volumes/main/adtech_raw/adtech-raw"

# Create sub-directories for each event type
EVENT_TYPES = [
    "bid_requests", "impressions", "clicks", "conversions",
    "campaign_cdc", "refunds", "billing_records", "ledger_entries"
]

for et in EVENT_TYPES:
    path = f"{VOLUME_PATH}/{et}"
    os.makedirs(path, exist_ok=True)

# Write and delete a test file to confirm write access
test_file = f"{VOLUME_PATH}/.write_test"
try:
    with open(test_file, "w") as f:
        f.write("ok")
    os.remove(test_file)
    print(f"✓ Volume is writable: {VOLUME_PATH}")
    print(f"✓ Event type dirs created: {EVENT_TYPES}")
except PermissionError:
    print("✗ PERMISSION DENIED on Volume.")
    print("  Fix: run the grant command below in a SQL cell:")
    print()
    print("  GRANT WRITE VOLUME ON VOLUME main.adtech_raw.`adtech-raw`")
    print("  TO `jatin.jangid.104@gmail.com`;")
except Exception as e:
    print(f"✗ Unexpected error: {e}")

In [0]:
# Restart Python to pick up the fixed source files
dbutils.library.restartPython()

In [0]:
from orchestrator import GeneratorOrchestrator
import time

print("Initialising orchestrator (building user pool + campaigns)...")
t0 = time.time()
orchestrator = GeneratorOrchestrator(config, validator)
elapsed = time.time() - t0

print(f"Orchestrator ready in {elapsed:.1f}s")
print(f"Generators  : {[g.EVENT_TYPE for g in orchestrator._generators]}")
print(f"Output path : {config.platform.output_base_path}")

In [0]:
# For your first run, use max_iterations=20 to verify everything works.
# Change to None for continuous infinite streaming.

orchestrator.run(max_iterations=20)

In [0]:
import os

VOLUME_PATH = "/Volumes/main/adtech_raw/adtech-raw"

print(f"{'Event Type':<25} {'Files':>6} {'Records':>10}")
print("─" * 45)

total_records = 0
for et in ["bid_requests","impressions","clicks","conversions",
           "campaign_cdc","refunds","billing_records","ledger_entries"]:
    path = f"{VOLUME_PATH}/{et}"
    if not os.path.exists(path):
        print(f"{et:<25} {'no dir':>6}")
        continue
    files, records = 0, 0
    for root, _, fs in os.walk(path):
        for fname in fs:
            if fname.endswith(".json"):
                files += 1
                try:
                    with open(os.path.join(root, fname)) as f:
                        records += sum(1 for _ in f)
                except Exception:
                    pass
    total_records += records
    print(f"{et:<25} {files:>6} {records:>10,}")

print("─" * 45)
print(f"{'TOTAL':<25} {'':>6} {total_records:>10,}")